# DSW 2023 experiment: CatBoost 5-fold stratified CV baseline (accuracy)

In [27]:
import numpy as np # linear algebra
import pandas as pd # data processing
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

import keras
from keras.models import Sequential
from keras.layers import Dropout, BatchNormalization
from keras.layers import Dense
from catboost import CatBoostClassifier

from sklearn.preprocessing import LabelEncoder  
import os

DATA_DIR = "../data"
DATA_PATH = os.path.join(DATA_DIR, "telco_customer_churn_adapted_v2.xlsx")

In [2]:
train_df = pd.read_excel(DATA_PATH)

In [4]:
train_df

,Customer ID,Tenure Months,Location,Device Class,Games Product,Music Product,Education Product,Call Center,Video Product,Use MyApp,Payment Method,Monthly Purchase (Thou. IDR),Churn Label,Longitude,Latitude,CLTV (Predicted Thou. IDR)
0,0,2,Jakarta,Mid End,Yes,Yes,No,No,No,No,Digital Wallet,70.005,Yes,106.816666,-6.2,4210.7
1,1,2,Jakarta,High End,No,No,No,No,No,No,Pulsa,91.910,Yes,106.816666,-6.2,3511.3
2,2,8,Jakarta,High End,No,No,Yes,No,Yes,Yes,Pulsa,129.545,Yes,106.816666,-6.2,6983.6
3,3,28,Jakarta,High End,No,No,Yes,Yes,Yes,Yes,Pulsa,136.240,Yes,106.816666,-6.2,6503.9
4,4,49,Jakarta,High End,No,Yes,Yes,No,Yes,Yes,Debit,134.810,Yes,106.816666,-6.2,6942.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,7038,72,Jakarta,Low End,No internet service,No internet service,No internet service,No,No internet service,No internet service,Debit,27.495,No,106.816666,-6.2,6897.8
7039,7039,24,Jakarta,Mid End,Yes,No,Yes,Yes,Yes,Yes,Digital Wallet,110.240,No,106.816666,-6.2,2782.0
7040,7040,72,Jakarta,High End,No,Yes,Yes,No,Yes,Yes,Credit,134.160,No,106.816666,-6.2,7228.0
7041,7041,11,Jakarta,Mid End,Yes,No,No,No,No,No,Pulsa,38.480,No,106.816666,-6.2,3630.9


In [7]:
def preprocessing(df):
    df = df.copy()
    df.loc[df["Games Product"] != "No internet service", "internet_service"] = 1
    df["internet_service"].fillna(0, inplace=True)
    df["Location"] = df["Location"].map({"Jakarta": 1, "Bandung": 0})
    df["Device Class"] = df["Device Class"].map({"High End": 2, "Mid End": 1, "Low End": 0})
    df["Games Product"] = df["Games Product"].map({"Yes": 1, "No": 0, "No internet service": 0})
    df["Music Product"] = df["Music Product"].map({"Yes": 1, "No": 0, "No internet service": 0})
    df["Education Product"] = df["Education Product"].map({"Yes": 1, "No": 0, "No internet service": 0})
    df["Call Center"] = df["Call Center"].map({"Yes": 1, "No": 0})
    df["Video Product"] = df["Video Product"].map({"Yes": 1, "No": 0, "No internet service": 0})
    df["Use MyApp"] = df["Use MyApp"].map({"Yes": 1, "No": 0, "No internet service": 0})
    df = pd.get_dummies(df, columns=["Payment Method"], prefix="pm")
    return df

In [23]:
le = LabelEncoder()
y = train_df['Churn Label']
y= le.fit_transform(y)
X = train_df.drop(['Churn Label', 'Customer ID'], axis=1)
X = preprocessing(X)

In [16]:
X.columns

Index(['Customer ID', 'Tenure Months', 'Location', 'Device Class',
       'Games Product', 'Music Product', 'Education Product', 'Call Center',
       'Video Product', 'Use MyApp', 'Monthly Purchase (Thou. IDR)',
       'Longitude', 'Latitude', 'CLTV (Predicted Thou. IDR)',
       'internet_service', 'pm_Credit', 'pm_Debit', 'pm_Digital Wallet',
       'pm_Pulsa'],
      dtype='object')

In [28]:
n_fold = 5
folds = StratifiedKFold(n_splits=n_fold, random_state = 42, shuffle=True)
splits = folds.split(X, y)
accuracy_total = 0
for count , (train_index, valid_index) in enumerate(splits):
    X_tr, X_val = X.iloc[train_index], X.iloc[valid_index]
    y_tr, y_val = y[train_index], y[valid_index]
    cb = CatBoostClassifier()
    cb.fit(X_tr, y_tr, verbose=250)
    y_pred = cb.predict(X_val)
    accuracy = accuracy_score(y_val, y_pred)
    accuracy_total += accuracy

# Final Result
print(f"Baseline Score: {accuracy_total/n_fold}")

Learning rate set to 0.021554
0:	learn: 0.6791424	total: 4.08ms	remaining: 4.08s
250:	learn: 0.3887153	total: 837ms	remaining: 2.5s
500:	learn: 0.3609049	total: 1.87s	remaining: 1.86s
750:	learn: 0.3353024	total: 2.86s	remaining: 949ms
999:	learn: 0.3150333	total: 3.9s	remaining: 0us
Learning rate set to 0.021554
0:	learn: 0.6789648	total: 4.3ms	remaining: 4.3s
250:	learn: 0.3869635	total: 1.02s	remaining: 3.05s
500:	learn: 0.3589169	total: 1.94s	remaining: 1.93s
750:	learn: 0.3328781	total: 2.84s	remaining: 941ms
999:	learn: 0.3103969	total: 3.73s	remaining: 0us
Learning rate set to 0.021554
0:	learn: 0.6786219	total: 3.4ms	remaining: 3.4s
250:	learn: 0.3888344	total: 925ms	remaining: 2.76s
500:	learn: 0.3598877	total: 1.82s	remaining: 1.81s
750:	learn: 0.3368659	total: 2.69s	remaining: 892ms
999:	learn: 0.3142332	total: 3.57s	remaining: 0us
Learning rate set to 0.021556
0:	learn: 0.6783775	total: 2.96ms	remaining: 2.96s
250:	learn: 0.3873033	total: 929ms	remaining: 2.77s
500:	learn: 